In [ ]:
!pip install reward_utils

ERROR: Could not find a version that satisfies the requirement reward_utils (from versions: none)
ERROR: No matching distribution found for reward_utils


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from transformers import BartTokenizer, BartForConditionalGeneration

# Assuming reward_utils.py is in the same directory or accessible via PYTHONPATH
from reward_utils import compute_reward


class SynthesizerAgent(nn.Module):
    """
    An A2C agent for abstractive summarization using BART.
    BART's decoder acts as the policy (actor), generating summaries.
    A separate critic network estimates the value of the input state.
    """

    def __init__(self, model_name: str = 'facebook/bart-base', lr: float = 3e-5,
                 device: str = None, max_output_len: int = 120, max_input_len: int = 512):
        """
        Initializes the SynthesizerAgent.

        Args:
            model_name (str): Name of the pre-trained BART model (e.g., 'facebook/bart-base').
            lr (float): Learning rate for the Adam optimizer.
            device (str, optional): Device to run the model on ('cuda' or 'cpu').
                                    Defaults to 'cuda' if available, else 'cpu'.
            max_output_len (int): Maximum token length for the generated summary.
            max_input_len (int): Maximum token length for the input text.
        """
        super(SynthesizerAgent, self).__init__()
        self.device = device if device else ('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"SynthesizerAgent will use device: {self.device}")

        self.tokenizer = BartTokenizer.from_pretrained(model_name)
        self.model = BartForConditionalGeneration.from_pretrained(model_name).to(self.device)

        self.max_output_len = max_output_len
        self.max_input_len = max_input_len

        # A2C components
        # Critic network takes the BART encoder's output dimension as input.
        self.critic = nn.Linear(self.model.config.d_model, 1).to(self.device)

        # Optimizer for both BART (actor) and critic parameters. AdamW is generally preferred.
        self.optimizer = optim.AdamW(list(self.model.parameters()) + list(self.critic.parameters()), lr=lr)

    @torch.no_grad()  # This method is for inference/evaluation, no gradients needed
    def generate_summary(self, input_text: str, max_length: int = None, num_beams: int = 4) -> str:
        """
        Generates a summary using BART's beam search for inference or evaluation.

        Args:
            input_text (str): The input text (e.g., from Simplifier agent).
            max_length (int, optional): Max length for the generated summary.
                                        Overrides self.max_output_len if provided.
            num_beams (int): Number of beams for beam search.

        Returns:
            str: The generated summary.
        """
        actual_max_output_len = max_length if max_length is not None else self.max_output_len

        inputs = self.tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=self.max_input_len
        ).to(self.device)

        summary_ids = self.model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=actual_max_output_len,
            num_beams=num_beams,
            early_stopping=True,
            no_repeat_ngram_size=2  # Good practice to reduce repetition
        )
        return self.tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    def train_step_batched(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, labels: torch.Tensor,
                           gamma: float = 0.99, entropy_coef: float = 0.01):
        """
        Performs one A2C training step for a BATCH of data.

        Args:
            input_ids (torch.Tensor): Batched input_ids from tokenizer.
            attention_mask (torch.Tensor): Batched attention_mask from tokenizer.
            labels (torch.Tensor): Batched reference summary token IDs (for reward calculation).
                                   -100 values will be replaced by pad_token_id for decoding.
            gamma (float): Discount factor for future rewards (not fully applied in this simple A2C).
            entropy_coef (float): Coefficient for the entropy regularization term (not used in this version).

        Returns:
            Tuple[List[str], torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
                - generated_summaries (List[str]): The summaries generated during this training step.
                - rewards (torch.Tensor): The rewards obtained for generated summaries.
                - policy_loss (torch.Tensor): The calculated policy loss (actor).
                - value_loss (torch.Tensor): The calculated value loss (critic).
                - total_loss (torch.Tensor): The combined policy and value loss.
        """
        # Note: self.optimizer.zero_grad() and self.optimizer.step() are handled
        # by the outer A2CTrainer for gradient accumulation.

        # --- 1. Actor: Generate summary stochastically and get log probabilities ---
        generated_outputs = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=self.max_output_len,
            do_sample=True,
            top_k=50,  # Example sampling strategy
            temperature=1.0,  # Controls randomness
            num_return_sequences=1,  # Generate one summary per input example for each input
            output_scores=True,
            return_dict_in_generate=True
        )
        generated_ids = generated_outputs.sequences  # (batch_size, sequence_length)

        # Decode generated summaries for reward calculation
        generated_summaries = [
            self.tokenizer.decode(g_ids, skip_special_tokens=True) for g_ids in generated_ids
        ]

        # Decode reference summaries for reward calculation
        # Replace -100 in labels with tokenizer.pad_token_id for proper decoding
        temp_labels = labels.clone()  # Clone to avoid modifying original labels
        temp_labels[temp_labels == -100] = self.tokenizer.pad_token_id
        reference_summaries = [
            self.tokenizer.decode(l_ids, skip_special_tokens=True) for l_ids in temp_labels
        ]

        # Calculate log-probabilities for each generated sequence in the batch
        log_probs_batch = []
        for i in range(generated_ids.shape[0]):  # Iterate over each example in the batch
            log_prob_of_single_sequence = 0.0

            # generated_outputs.scores is a list of tensors, where each tensor is (batch_size, vocab_size)
            # Its length is `sequence_length - 1` (excluding the initial decoder_start_token)

            for j, logits_at_step_j in enumerate(generated_outputs.scores):
                if j + 1 < generated_ids.shape[1]:  # Check if sampled_token_id exists for this step
                    sampled_token_id = generated_ids[i, j + 1]  # +1 to skip decoder_start_token_id (BOS)
                    # Get log_softmax for the current step's logits for the i-th example
                    log_prob_token = F.log_softmax(logits_at_step_j[i].unsqueeze(0), dim=-1)[:, sampled_token_id].item()
                    log_prob_of_single_sequence += log_prob_token
                else:
                    # This case should ideally not be hit if lengths align, but as a safeguard
                    pass # Or log a warning if needed
            log_probs_batch.append(log_prob_of_single_sequence)

        # Convert list of log-probs to a tensor, shape (batch_size,)
        log_prob_of_generated_sequence_tensor = torch.tensor(log_probs_batch, device=self.device)


        # --- 2. Calculate Rewards for the batch ---
        rewards = torch.tensor([compute_reward(gen_s, ref_s)
                                for gen_s, ref_s in zip(generated_summaries, reference_summaries)],
                               device=self.device)

        # --- 3. Critic: Estimate Value (Baseline) ---
        # Get encoder outputs for the input context to feed to the critic
        encoder_outputs = self.model.model.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state  # (batch_size, sequence_length, hidden_size)

        # Pooled output for the critic (e.g., mean pooling across sequence length)
        pooled_output = encoder_outputs.mean(dim=1)  # (batch_size, hidden_size)
        baseline_value = self.critic(pooled_output).squeeze(-1)  # (batch_size,)

        # --- 4. Calculate Advantage ---
        # Advantage is the difference between actual reward and critic's baseline.
        # Detach baseline to prevent gradients from flowing through the value head to the actor.
        advantage = rewards - baseline_value.detach()

        # --- 5. A2C Loss Calculation ---
        # Policy Loss (Actor Loss): Maximize expected reward using policy gradients
        # Negative sign because optimizers minimize loss.
        policy_loss = - (log_prob_of_generated_sequence_tensor * advantage).mean()  # Mean over batch

        # Value Loss (Critic Loss): MSE between predicted value and actual reward (target)
        value_loss = F.mse_loss(baseline_value, rewards)  # MSE over batch

        # Total Loss: Combine policy and value losses with a weighting factor for value loss
        total_loss = policy_loss + 0.5 * value_loss

        # Backpropagate the total loss.
        # Gradients are accumulated if `optimizer.zero_grad()` is not called before this.
        total_loss.backward()

        return generated_summaries, rewards, policy_loss, value_loss, total_loss

# Example usage (for testing the module independently if needed)
if __name__ == "__main__":
    # If testing directly, ensure reward_utils is in your Python path or dummy
    # function is defined as in the previous example.
    try:
        from reward_utils import compute_reward
    except ImportError:
        print("Warning: reward_utils.py not found. Using a dummy compute_reward function.")
        def compute_reward(generated_summary: str, reference_summary: str) -> float:
            return 0.1 # Dummy reward

    print("--- Initializing SynthesizerAgent for testing ---")
    agent = SynthesizerAgent(model_name='facebook/bart-base')
    print(f"Synthesizer Agent initialized on {agent.device}")

    # Prepare dummy batched inputs for train_step_batched
    test_input_texts = [
        "The quick brown fox jumps over the lazy dog.",
        "Quantum computing promises to revolutionize information processing."
    ]
    test_reference_summaries = [
        "A fast fox jumps over a sleepy dog.",
        "Quantum computing will transform data processing."
    ]

    # Tokenize these dummy inputs
    inputs = agent.tokenizer(
        test_input_texts,
        return_tensors="pt",
        padding='max_length',
        truncation=True,
        max_length=agent.max_input_len
    ).to(agent.device)

    # Tokenize reference summaries to get labels
    labels = agent.tokenizer(
        text_target=test_reference_summaries,
        return_tensors="pt",
        padding='max_length',
        truncation=True,
        max_length=agent.max_output_len
    ).input_ids.to(agent.device)

    # Replace padding token id with -100 for loss calculation
    labels[labels == agent.tokenizer.pad_token_id] = -100

    print("\n--- Testing train_step_batched ---")
    # For a single test, we can call zero_grad and step here directly
    agent.optimizer.zero_grad()
    try:
        gen_summaries, rewards_tensor, p_loss, v_loss, t_loss = agent.train_step_batched(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            labels=labels
        )
        agent.optimizer.step() # Step the optimizer after the backward pass

        print(f"Generated Summaries (Batch 1): {gen_summaries[0]}")
        print(f"Generated Summaries (Batch 2): {gen_summaries[1]}")
        print(f"Rewards: {rewards_tensor.tolist()}")
        print(f"Policy Loss: {p_loss.item():.4f}")
        print(f"Value Loss: {v_loss.item():.4f}")
        print(f"Total Loss: {t_loss.item():.4f}")
        print(f"BART model parameters with gradients: {any(p.grad is not None for p in agent.model.parameters())}")
        print(f"Critic parameters with gradients: {any(p.grad is not None for p in agent.critic.parameters())}")

    except Exception as e:
        print(f"An error occurred during train_step_batched: {e}")
        import traceback
        traceback.print_exc() # Print full traceback for debugging

    print("\n--- Testing generate_summary (Inference) ---")
    agent.eval() # Set to evaluation mode
    test_input_text = "The new smartphone features an advanced camera system and longer battery life."
    inferred_summary = agent.generate_summary(test_input_text)
    print(f"Input for Inference: {test_input_text}")
    print(f"Generated Summary: {inferred_summary}")

ModuleNotFoundError: No module named 'reward_utils'